# DataOps Pipeline — Validation

Validates the DataOps pipeline output by querying the Glue Catalog
table `campaign_results` via Athena and checking row counts,
schema, and sample data.

In [ ]:
# Parameters
catalog_db = "bank_mktg_dev"
table_name = "campaign_results"
region = "us-east-1"

In [ ]:
import boto3
import time
import os

catalog_db = catalog_db or os.environ.get('ATHENA_DB_NAME', 'bank_mktg_dev')
region = region or os.environ.get('AWS_DEFAULT_REGION', 'us-east-1')

athena = boto3.client('athena', region_name=region)
glue = boto3.client('glue', region_name=region)

print(f'Database: {catalog_db}')
print(f'Table:    {table_name}')
print(f'Region:   {region}')

## 1. Check Table Exists in Glue Catalog

In [ ]:
try:
    resp = glue.get_table(DatabaseName=catalog_db, Name=table_name)
    table = resp['Table']
    print(f'Table found: {catalog_db}.{table_name}')
    print(f'Location:    {table["StorageDescriptor"]["Location"]}')
    print(f'Format:      {table["StorageDescriptor"]["InputFormat"]}')
    print(f'Columns:     {len(table["StorageDescriptor"]["Columns"])}')
    print()
    for col in table['StorageDescriptor']['Columns']:
        print(f'  {col["Name"]:25s} {col["Type"]}')
except glue.exceptions.EntityNotFoundException:
    raise RuntimeError(f'Table {catalog_db}.{table_name} not found')

## 2. Row Count

In [ ]:
def run_athena_query(query, database, region):
    """Run Athena query and return results."""
    resp = athena.start_query_execution(
        QueryString=query,
        QueryExecutionContext={'Database': database},
        ResultConfiguration={'OutputLocation': f's3://aws-athena-query-results-{region}/'}
    )
    qid = resp['QueryExecutionId']
    while True:
        status = athena.get_query_execution(QueryExecutionId=qid)
        state = status['QueryExecution']['Status']['State']
        if state in ('SUCCEEDED', 'FAILED', 'CANCELLED'):
            break
        time.sleep(2)
    if state != 'SUCCEEDED':
        raise RuntimeError(f'Query failed: {state}')
    results = athena.get_query_results(QueryExecutionId=qid)
    return results

results = run_athena_query(f'SELECT COUNT(*) as cnt FROM {table_name}', catalog_db, region)
row_count = int(results['ResultSet']['Rows'][1]['Data'][0]['VarCharValue'])
print(f'Row count: {row_count}')
assert row_count > 0, 'Table is empty'

## 3. Sample Data

In [ ]:
results = run_athena_query(f'SELECT * FROM {table_name} LIMIT 5', catalog_db, region)
headers = [col['VarCharValue'] for col in results['ResultSet']['Rows'][0]['Data']]
print(' | '.join(headers))
print('-' * 80)
for row in results['ResultSet']['Rows'][1:]:
    vals = [d.get('VarCharValue', '') for d in row['Data']]
    print(' | '.join(vals))

## 4. Summary

In [ ]:
print('=' * 60)
print('DATAOPS VALIDATION SUMMARY')
print('=' * 60)
print(f'  Database:  {catalog_db}')
print(f'  Table:     {table_name}')
print(f'  Rows:      {row_count}')
print(f'  Columns:   {len(headers)}')
print(f'  Status:    PASSED')
print('=' * 60)